In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from tqdm import tqdm # Para la barra de progreso

# --- 1. PREPARACIÓN RÁPIDA DE DATOS ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

df = pd.read_csv('../data/HAM10000_metadata_prepared.csv')
label_map = {clase: idx for idx, clase in enumerate(df['dx'].unique())}
df['label'] = df['dx'].map(label_map)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

class SkinCancerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.dataframe)
    def __getitem__(self, idx):
        img = Image.open(self.dataframe.loc[idx, 'image_path']).convert('RGB')
        label = self.dataframe.loc[idx, 'label']
        if self.transform: img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(SkinCancerDataset(train_df, transform=train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(SkinCancerDataset(val_df, transform=val_transform), batch_size=32, shuffle=False)

# --- 2. CONFIGURACIÓN DEL MODELO (TRANSFER LEARNING) ---
# Descargamos ResNet18 pre-entrenada
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Cambiamos la última capa (fc) para que en vez de 1000 salidas, tenga 7 (nuestras clases)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 7)
model = model.to(device)

# --- 3. BALANCEO DE CLASES Y OPTIMIZADOR ---
# Calculamos los pesos: las clases con menos imágenes tendrán un peso mayor
class_weights = compute_class_weight('balanced', classes=np.unique(train_df['label']), y=train_df['label'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Función de pérdida (CrossEntropyLoss) con los pesos aplicados
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizador (Adam)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- 4. BUCLE DE ENTRENAMIENTO ---
epochs = 5 # Vamos a hacer 5 vueltas completas al dataset para probar
best_acc = 0.0

print("\nIniciando entrenamiento...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    # Barra de progreso para el entrenamiento
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()         # Limpiar cálculos anteriores
        outputs = model(images)       # Pasar las imágenes por la IA
        loss = criterion(outputs, labels) # Calcular el error
        loss.backward()               # Calcular cómo corregir el error (Backpropagation)
        optimizer.step()              # Actualizar los "conocimientos" de la IA
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': running_loss/len(train_loader)})
        
    # --- 5. FASE DE VALIDACIÓN ---
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad(): # no aprende, solo se evalúa
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    epoch_acc = 100 * correct / total
    print(f"Precisión en Validación: {epoch_acc:.2f}%\n")
    
    # Guardar el mejor modelo
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), '../data/best_model.pth')

print(f"Entrenamiento finalizado. Mejor precisión: {best_acc:.2f}%")
print("Mejor modelo guardado en 'data/best_model.pth'")

Entrenando en: cuda


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\polor/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:02<00:00, 16.4MB/s]



Iniciando entrenamiento...


Epoch 1/5 [Train]: 100%|██████████| 251/251 [02:34<00:00,  1.62it/s, loss=1.58] 


Precisión en Validación: 26.21%



...

Precisión en Validación: 51.32%



Epoch 4/5 [Train]: 100%|██████████| 251/251 [01:31<00:00,  2.73it/s, loss=1.17] 


Precisión en Validación: 51.82%



Epoch 5/5 [Train]: 100%|██████████| 251/251 [01:21<00:00,  3.08it/s, loss=1.11] 


Precisión en Validación: 45.33%

Entrenamiento finalizado. Mejor precisión: 51.82%
Mejor modelo guardado en 'data/best_model.pth'
